# 01 — Data acquisition, audit and exploratory analysis (laptop)

Runs on the RTX 3050 laptop (4 GB VRAM). Everything here is either CPU-bound or
small enough to fit the card. The heavy sweeps live in `02_kaggle_experiments.ipynb`.

**What this notebook decides:** the input resolution for the study. That single
choice determines whether the experiments measure the *method* or measure the
*resize* — see [docs/10-datasets.md](../docs/10-datasets.md) and
[docs/01-charter-and-protocol.md](../docs/01-charter-and-protocol.md) §4.1.

**Order matters.** Fit is never called before the split audit passes. A leaked
split does not raise an error, it produces a *better* score, so it has to be
ruled out mechanically before any number is generated.

---

Dataset: **VisA** (Amazon), CC BY 4.0. No images are redistributed by this
repository. Attribution in [ATTRIBUTION.md](../ATTRIBUTION.md).

## 0. Environment

In [ ]:
import sys, subprocess, pathlib

REPO = pathlib.Path.cwd()
if not (REPO / 'src' / 'inspector').exists():
    REPO = REPO.parent  # running from notebooks/
sys.path.insert(0, str(REPO / 'src'))

from inspector.utils.env import capture

env = capture()
print('git      :', env['git_sha'], '(dirty)' if env['dirty'] else '(clean)')
print('device   :', env['device'].get('gpu_name', 'CPU only'),
      f"{env['device'].get('gpu_total_mb', 0)} MB")
print('torch    :', env['libraries'].get('torch', 'not installed'))
print()
print('A dirty tree tags every run dirty, and dirty runs may not enter the')
print('headline table (docs/06 section 2). Commit before a reportable run.')

## 1. Acquire VisA

1.8 GB over HTTPS, no account. The fetch records a SHA-256 manifest so a
silently truncated or updated archive is detected rather than trained on.

In [ ]:
from inspector.fetch import fetch, verify

DATA_RAW = REPO / 'data' / 'raw'
manifest = fetch('visa', DATA_RAW, extract=True)

VISA_ROOT = pathlib.Path(manifest.extracted_to)
print('root     :', VISA_ROOT)
print('action   :', manifest.action)
for name, digest in manifest.files.items():
    print(f'  {name:26s} sha256={digest[:16]}')
print()
print('license  :', manifest.license_note)

In [ ]:
# Re-run this after any sync to Drive or Kaggle. Sync corruption is silent,
# and finding it a week later costs a week.
for name, ok in verify('visa', DATA_RAW).items():
    print(f"  {'OK  ' if ok else 'FAIL'} {name}")

## 2. The official one-class split

`split_csv/1cls.csv` is authoritative. Inventing a split would produce numbers
comparable with nothing, so the loader treats a missing CSV as an error rather
than falling back on directory scanning.

In [ ]:
from inspector.data.visa import VISA_GROUPS, summarize
from inspector.postproc import min_achievable_fpr

counts = summarize(VISA_ROOT)
print(f"{'category':<13}{'group':<20}{'train':>7}{'test_N':>8}{'test_A':>8}"
      f"{'val@15%':>9}{'minFPR':>8}")
print('-' * 73)
for cat, c in counts.items():
    n_val = max(4, round(c['train_normal'] * 0.15))
    print(f"{cat:<13}{c['group']:<20}{c['train_normal']:>7}{c['test_normal']:>8}"
          f"{c['test_anomaly']:>8}{n_val:>9}{min_achievable_fpr(n_val):>7.2%}")

### Read the `minFPR` column

A threshold set as the k-th largest of `n` validation scores is exceeded by a
fresh normal sample with probability `k/(n+1)`, so the achievable false-alarm
floor is `1/(n+1)` — see [ADR-7](../docs/07-risks-and-decisions.md).

On VisA that floor lands **on both sides of 1%**: the large categories
(`candle`, `macaroni*`, `pcb*`) can be calibrated to 1%, the small ones
(`cashew`, `fryum`, `chewinggum`, `pipe_fryum`) cannot. The same protocol is
therefore achievable on some categories and not others *within one dataset*,
which is why the operating point is reported per category rather than as one
headline FPR.

The floor depends on the 15% carve fraction, which is our choice, not VisA's.
Raising it buys a lower floor at the cost of training images — an ablation worth
running rather than a default worth assuming.

## 3. Study categories

One from each of VisA's three structural groups, so each fails for a different
reason. Three PCBs would test one difficulty three times.

| category | group | axis it stresses |
|---|---|---|
| `pcb1` | complex structure | resolution / small defects |
| `macaroni2` | multiple instances | false positives from normal variation |
| `capsules` | multiple instances | optics: transparent, reflective |

In [ ]:
CATEGORIES = ['pcb1', 'macaroni2', 'capsules']
CONTROL = 'cashew'  # easiest group; a method that fails here is broken

from inspector.data import ensure_validation, load_category

indices = {}
for cat in CATEGORIES:
    idx = load_category(VISA_ROOT, cat, layout='visa')
    idx, assignment = ensure_validation(idx, val_fraction=0.15, seed=0)
    indices[cat] = idx
    print(f'{cat:<11} ' + '  '.join(
        f'{k}={len(v)}(N{v.n_normal}/A{v.n_anomalous})' for k, v in sorted(idx.items())))
    if assignment:
        print(f'{"":<11} validation carved from train, seed={assignment.seed} '
              f'digest={assignment.digest[:12]}')

## 4. Split integrity — run before anything is fitted

Protocol rules L1-L7. L1 compares SHA-256 of file **bytes**, not filenames: the
failure mode that matters is the same capture appearing in two splits under
different names, which a filename check would miss entirely.

In [ ]:
from inspector.data.integrity import audit

all_ok = True
for cat, idx in indices.items():
    print(f'--- {cat} ---')
    for report in audit(idx):
        print(' ', report)
        all_ok &= report.ok

assert all_ok, 'split integrity failed: fix before generating any number'
print('\nall integrity checks passed')

## 5. Exploratory analysis — the resolution decision

Measures every connected defect region, then computes what each candidate
resolution does to the **smallest** defects. The percentiles matter more than
the median here: the median defect is rarely the one a model misses.

In [ ]:
from inspector.eda import analyse_category
from inspector.report import write_eda_report

CANDIDATES = (128, 224, 256, 320, 448, 512)
reports = []
for cat, idx in indices.items():
    print(f'analysing {cat} ...')
    reports.append(analyse_category(idx, test_split='test', candidates=CANDIDATES))

md_path, json_path = write_eda_report(reports, REPO / 'reports' / 'eda_visa')
print('\nreport ->', md_path)

In [ ]:
for r in reports:
    s = r.region_summary
    print(f'=== {r.category}  native {r.native_size[0]}x{r.native_size[1]} '
          f'({r.aspect_ratio:.2f}:1) ===')
    print(f'  regions            : {int(s["n_regions"])} over {int(s["n_images"])} images')
    print(f'  area %% of image    : p1 {s["area_fraction_p1"]:.4%}  '
          f'median {s["area_fraction_median"]:.4%}  max {s["area_fraction_max"]:.3%}')
    print(f'  thin dimension px  : p1 {s["min_extent_p1"]:.1f}  median {s["min_extent_median"]:.1f}')
    print(f'  local contrast     : p5 {s["local_contrast_p5"]:.1f}  median {s["local_contrast_median"]:.1f} grey levels')
    print()

In [ ]:
print(f"{'category':<12}{'long side':>10}{'output':>12}{'MP':>7}{'p5 thin':>9}{'<1px':>8}   verdict")
print('-' * 70)
for r in reports:
    for v in r.resolution_aspect:
        print(f'{r.category:<12}{v.long_side:>10}{v.out_width}x{v.out_height:<8}'
              f'{v.megapixels:>6.2f}{v.p5_min_extent_after:>8.2f}p{v.frac_regions_below_1px:>7.1%}   {v.verdict}')
    d = r.recommended
    print(f'{"":<12} -> {"TILING REQUIRED" if d.requires_tiling else d.chosen.long_side}')
    print(f'{"":<12}    {d.rationale}')
    print()

### The cost of a blind square resize

Shown separately because `resize(256, 256)` is the reflex, and on a non-square
frame it squashes one axis harder than the other. A defect's thin dimension is
hit by whichever axis is squashed hardest, so the damage is worse than the
megapixel count suggests.

In [ ]:
for r in reports:
    a = {v.long_side: v for v in r.resolution_aspect}
    s = {v.long_side: v for v in r.resolution_square}
    print(f'=== {r.category} ===')
    print(f"{'long':>6}{'aspect p5':>12}{'square p5':>12}{'aspect MP':>11}{'square MP':>11}")
    for k in sorted(set(a) & set(s)):
        print(f'{k:>6}{a[k].p5_min_extent_after:>11.2f}p{s[k].p5_min_extent_after:>11.2f}p'
              f'{a[k].megapixels:>11.2f}{s[k].megapixels:>11.2f}')
    print()

## 6. Visual check of the defect population

Always look at the data before trusting a statistic about it. A contact sheet
catches a mask misalignment or an empty ground truth that no percentile would.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from inspector.data.transforms import load_image, load_mask
from inspector.data.core import ANOMALOUS

fig, axes = plt.subplots(len(CATEGORIES), 4, figsize=(13, 3.4 * len(CATEGORIES)))
for row, cat in enumerate(CATEGORIES):
    anomalous = indices[cat]['test'].filter(label=ANOMALOUS)
    for col in range(4):
        sample = anomalous[col * max(1, len(anomalous) // 4)]
        image = load_image(sample.image_path)
        mask = load_mask(sample.mask_path)
        overlay = image.copy()
        overlay[mask] = (0.45 * overlay[mask] + 0.55 * np.array([255, 40, 40])).astype('uint8')
        ax = axes[row, col]
        ax.imshow(overlay)
        ax.set_title(f'{cat}  defect {mask.mean():.3%} of frame', fontsize=8)
        ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Tier 0 floors

Four models whose job is to fail, and to prove the metric code is right before
anything expensive runs (Gate G2):

* `random` must land at image AUROC ~0.5 and AU-PRO@L ~L/2. If it does not, the
  metric is broken and every later number is wrong.
* `mean_intensity` / `histogram` reveal a category separable by exposure alone.
  If one of them scores well, a deep model's success there measures the lighting.
* `pixel_pca` is the honest floor for the autoencoder: a conv AE that cannot beat
  linear PCA on raw pixels has learned nothing a linear projection did not.

In [ ]:
import tempfile
from inspector.data.transforms import ImageTransform
from inspector.models import build_model
from inspector.pipeline import run_experiment
from inspector.results import append_results, render_markdown
from inspector.utils.seed import seed_everything

RESOLUTION = 256   # revise from the section-5 table if it says otherwise
SEEDS = [0, 1, 2]
transform = ImageTransform(mode='aspect_preserving', long_side=RESOLUTION)

tier0 = []
for cat in CATEGORIES:
    for name in ['random', 'mean_intensity', 'histogram', 'pixel_pca']:
        for seed in SEEDS:
            seed_everything(seed)
            model = build_model(name, transform, seed=seed)
            with tempfile.TemporaryDirectory() as tmp:
                res = run_experiment(model, indices[cat], workdir=tmp, seed=seed,
                                     test_split='test')
            tier0.append(res)
    print(f'{cat} done')

append_results(tier0, REPO / 'reports' / 'results_visa.csv')
print(render_markdown(tier0))

In [ ]:
# Gate G2: the control must be at chance. Averaged over seeds and categories,
# because a single 200-image split still has meaningful variance.
import statistics

rnd = [r for r in tier0 if r.method == 'random']
print(f'random pixel AUROC   : {statistics.mean(r.pixel_auroc for r in rnd):.4f}  (expect 0.500)')
print(f'random AU-PRO@0.05   : {statistics.mean(r.aupro_005 for r in rnd):.4f}  (expect 0.025)')
print(f'random AU-PRO@0.30   : {statistics.mean(r.aupro_030 for r in rnd):.4f}  (expect 0.150)')
print()
for cat in CATEGORIES:
    trivial = [r for r in tier0 if r.category == cat and r.method in ('mean_intensity', 'histogram')]
    best = max(r.image_auroc for r in trivial)
    flag = '  <-- SEPARABLE BY A GLOBAL STATISTIC, report this prominently' if best > 0.8 else ''
    print(f'{cat:<12} best trivial image AUROC {best:.3f}{flag}')

## 8. PatchCore on the laptop

Fits inside 4 GB at 256² with `resnet18` and a 1% coreset. This is the
development configuration, not the reported one — the WideResNet50-2 reference
config and the ablation sweep belong on Kaggle.

There is no training here at all: a forward pass over the normal images, then a
coreset. That is why PatchCore is the right upgrade target — no training noise
to confound an ablation.

In [ ]:
import time, torch

laptop_runs = []
for cat in CATEGORIES:
    seed_everything(0)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    model = build_model('patchcore', transform, seed=0,
                        backbone='resnet18', layers=('layer2', 'layer3'),
                        coreset_ratio=0.01, batch_size=4, k=1)
    t0 = time.perf_counter()
    with tempfile.TemporaryDirectory() as tmp:
        res = run_experiment(model, indices[cat], workdir=tmp, seed=0, test_split='test')
    laptop_runs.append(res)
    print(res.summary())
    print('   ', {k: v for k, v in res.fit_extra.items()
                 if k in ('memory_bank_size', 'memory_bank_mb', 'peak_vram_mb', 'feature_grid')})

append_results(laptop_runs, REPO / 'reports' / 'results_visa.csv')

In [ ]:
print(render_markdown(tier0 + laptop_runs))

## 9. Qualitative check

The number and the picture must agree. A high AU-PRO with heatmaps that light up
the background means the metric is being satisfied by something other than the
defect, and that is worth knowing before the sweep runs.

In [ ]:
cat = CATEGORIES[0]
seed_everything(0)
model = build_model('patchcore', transform, seed=0, backbone='resnet18',
                    coreset_ratio=0.01, batch_size=4)
model.fit(indices[cat]['train'])

anomalous = indices[cat]['test'].filter(label=ANOMALOUS)
picks = [anomalous[i] for i in range(0, len(anomalous), max(1, len(anomalous) // 4))][:4]

fig, axes = plt.subplots(3, len(picks), figsize=(3.2 * len(picks), 9))
for col, sample in enumerate(picks):
    image = load_image(sample.image_path)
    mask = load_mask(sample.mask_path)
    pred = model.predict_sample(sample)
    axes[0, col].imshow(image);            axes[0, col].set_title('input', fontsize=9)
    axes[1, col].imshow(mask, cmap='gray'); axes[1, col].set_title('ground truth', fontsize=9)
    axes[2, col].imshow(image)
    axes[2, col].imshow(pred.anomaly_map, cmap='inferno', alpha=0.55)
    axes[2, col].set_title(f'score {pred.score:.3f}', fontsize=9)
    for row in range(3):
        axes[row, col].axis('off')
plt.tight_layout(); plt.show()

## 10. Hand-off to Kaggle

What stays here: acquisition, audit, EDA, Tier 0, and development-scale
PatchCore. What moves to `02_kaggle_experiments.ipynb`:

| workload | why not here |
|---|---|
| PatchCore with WideResNet50-2 at 320-512² | exceeds 4 GB |
| The ablation sweep (backbone x layers x resolution x coreset) | tens of GPU-hours |
| Autoencoder training, 3 seeds x 3 losses | hours of training |
| Robustness grid | 6 corruptions x 5 severities x 3 categories |

Record what this notebook produced in the compute ledger, then open notebook 02.

In [ ]:
ledger = REPO / 'reports' / 'compute_ledger.md'
ledger.parent.mkdir(parents=True, exist_ok=True)
with open(ledger, 'a', encoding='utf-8') as fh:
    fh.write(f'\n## laptop session\n')
    fh.write(f'- categories: {CATEGORIES}\n')
    fh.write(f'- tier0 runs: {len(tier0)}, patchcore runs: {len(laptop_runs)}\n')
    fh.write(f'- resolution: {RESOLUTION}, backbone: resnet18\n')
print('appended to', ledger)